# 6d · E2E entity-type canonicalization  (twin of nb4, for the single-call graph)

Runs the **same** type canonicalization as nb4 — *define → embed → retrieve → reuse-best-match-else-mint* —
but on the **e2e** single-call graph, which arrives with **free-form entity types**. For each distinct
`entities[].type` it retrieves candidates against **schema.org ∪ e2e-vocabulary**, lets the LLM reuse or
mint, then **relabels entities by id** (ids preserved, so the graph's `relations` stay valid — NO merging,
unlike nb4's within-type dedup, since the single call already resolved coreference).

- **Input:** `results/<id>/e2e/nary_graph.json` (from 6c).
- **Output:** `results/<id>/e2e/nary_graph_typed.json` (entities gain `type` = canonical label, plus
  `type_oie`, `type_source ∈ {schema.org, vocab}`, `type_vocab_id`, `type_parent`, …) +
  `e2e/entity_type_matching_review.json`.
- **Separate vocabulary:** grows `results/_global/e2e/type_vocabulary.json` — independent of the staged
  track's `type_vocabulary.json`, so the two methods' type economies stay comparable.

Then run **6e** to canonicalize the relations. e2e has no per-type definitions, so the embedding query is
the type label enriched with a few example mentions/aliases.

In [32]:
# --- resolve Phase2_LLM_Extraction root (works whether this notebook lives at the
# project root or under notebooks/<track>/) so results/ & dataset/ paths are stable.
from pathlib import Path
def _find_root(start: Path = None):
    start = (start or Path.cwd()).resolve()
    for p in (start, *start.parents):
        if (p / 'results').is_dir() and (p / 'dataset').is_dir():
            return p
    return start
ROOT = _find_root()
# make Path writes create missing parent dirs (results/ now has sub-folders like
# two_staged/1_entities/); guarded so re-running the cell never double-wraps.
import pathlib as _pl
if not getattr(_pl.Path.write_text, '_mkparents', False):
    _wt, _wb = _pl.Path.write_text, _pl.Path.write_bytes
    def write_text(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wt(self, *a, **k)
    def write_bytes(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wb(self, *a, **k)
    write_text._mkparents = write_bytes._mkparents = True
    _pl.Path.write_text, _pl.Path.write_bytes = write_text, write_bytes

In [33]:
# --- Config: e2e entity-type canonicalization (6d) --------------------------------------------------
import json, re, os, copy
from pathlib import Path
from collections import Counter
import numpy as np

SCHEMA_DIR  = ROOT / "schema_org_files"     # schema.org CLASS pool from nb1
RESULTS_DIR = ROOT / "results"              # reads e2e/nary_graph.json (6c)
# Which raw e2e graph to canonicalize. VARIANT="" -> default nary_graph.json; VARIANT="dedup"/"holistic"
# -> nary_graph_<VARIANT>.json. Inputs, outputs, review files AND vocab are all scoped by VARIANT so
# variants never overwrite or mix. 6e MUST use the SAME VARIANT.
VARIANT = "dedup"                                  # "" | "dedup" | "holistic" | ...
_sfx    = f"_{VARIANT}" if VARIANT else ""
VOCAB_PATH  = RESULTS_DIR / f"_global/e2e/type_vocabulary{_sfx}.json"   # SEPARATE per variant

MATCH_MODEL  = "intfloat/e5-base-v2"
GEMINI_MODEL = "gemini-2.5-flash"
EXPAND_MODE  = True                        # True: admit minted types (grow). False: restrict to schema.org u vocab.
TOP_K_MATCH  = 5
ALIGN_QUERY_STYLE = True                   # enrich each type query with example mentions/aliases

# Which articles to run on — and the order the e2e vocabulary grows along (order matters: incremental).
#SELECTED_ARTICLES = None                  # None -> every article that has e2e/nary_graph.json, sorted
SELECTED_ARTICLES = [ "art1", "art2", "art4", "art5"]

_available = sorted(p.parent.parent.name for p in RESULTS_DIR.glob(f"*/e2e/nary_graph{_sfx}.json"))
if SELECTED_ARTICLES is None:
    ARTICLE_ORDER = _available
else:
    missing = [a for a in SELECTED_ARTICLES if a not in _available]
    if missing:
        print("WARNING: skipping (no e2e/nary_graph.json):", missing)
    ARTICLE_ORDER = [a for a in SELECTED_ARTICLES if a in _available]
assert ARTICLE_ORDER, "No runnable articles (need 6c e2e/nary_graph.json). Check SELECTED_ARTICLES."
print("expand:", EXPAND_MODE, "| match:", MATCH_MODEL, "| articles:", ARTICLE_ORDER)

expand: True | match: intfloat/e5-base-v2 | articles: ['art1', 'art2', 'art4', 'art5']


In [34]:
# First run only (sentence-transformers pulls torch; google-genai for the LLM stages).
%pip install -q sentence-transformers google-genai numpy

Note: you may need to restart the kernel to use updated packages.


In [35]:
# Embedding model (loaded once). e5 needs "query: " on the query side and "passage: " on documents;
# both the OIE-type queries and every document (schema.org OR vocab) go through the same convention.
from sentence_transformers import SentenceTransformer
_MODEL = SentenceTransformer(MATCH_MODEL)

def _prefixes(name):
    n = name.lower()
    if "e5" in n:  return "query: ", "passage: "
    if "bge" in n: return "Represent this sentence for searching relevant passages: ", ""
    return "", ""
QP, DP = _prefixes(MATCH_MODEL)

def embed(texts, prefix):
    return np.asarray(_MODEL.encode([prefix + t for t in texts],
                                    normalize_embeddings=True, batch_size=64))
print("embedder ready:", MATCH_MODEL, "| query_prefix", repr(QP), "| doc_prefix", repr(DP))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

embedder ready: intfloat/e5-base-v2 | query_prefix 'query: ' | doc_prefix 'passage: '


In [36]:
# Schema.org document pool (fixed) — embedded once. Vocabulary documents are embedded per article
# inside retrieval, since the vocabulary grows during the run.
schema_pool   = json.loads((SCHEMA_DIR / "schema_classes_enriched.json").read_text())
SCHEMA_LABELS = {c["label"] for c in schema_pool}
schema_emb    = embed([c["embedding_text"] for c in schema_pool], DP)
print(len(schema_pool), "schema.org classes embedded")

KeyboardInterrupt: 

In [ ]:
# The growing vocabulary store. One entry per canonical minted type:
#   id, canonical_label, definition, parent (schema.org label or vocab label), aliases[] (= its synonyms),
#   provenance[], support, first_seen, and path (derived: Thing > ... > parent, stamped on every save).
def load_vocab():
    if VOCAB_PATH.exists():
        return json.loads(VOCAB_PATH.read_text())
    return {"types": [], "next_id": 1}

def save_vocab(vocab):
    _annotate_vocab_paths(vocab)                 # refresh each entry's derived `path` from its parent chain
    VOCAB_PATH.write_text(json.dumps(vocab, indent=2, ensure_ascii=False))

def vocab_embedding_text(t):
    parent = t.get("parent") or "Thing"
    also   = f" Also called {', '.join(t['aliases'])}." if t.get("aliases") else ""
    return f'{t["canonical_label"]} ({parent}): {t.get("definition", "")}{also}'

def _find_vocab_by_label(vocab, label):
    for t in vocab["types"]:
        if t["canonical_label"] == label or label in t.get("aliases", []):
            return t
    return None

def _is_vocab_label(vocab, label):
    return _find_vocab_by_label(vocab, label) is not None

def _bump(ent, aid, oie_type, group):
    ent["support"] = ent.get("support", 0) + 1
    ent.setdefault("provenance", []).append(
        {"article": aid, "oie_type": oie_type, "mentions": [e["text"] for e in group]})

In [ ]:
# Hook A — retrieve top-k candidates for each OIE type against schema.org u current vocabulary.
# Both schema.org AND vocab candidates expose a full ancestor `path` (Thing > ... > immediate parent),
# so the type-selection LLM can reach for any ancestor as a mint parent, symmetrically for either source.
_SCHEMA_BY_LABEL = {c["label"]: c for c in schema_pool}

def _schema_chain(label):
    """Top-down ancestor chain INCLUDING `label` for a schema.org class: [Thing, ..., label].
    (schema.org `path` lists ancestors only — its last element is the immediate parent.)"""
    c   = _SCHEMA_BY_LABEL.get(label)
    anc = [a for a in (c.get("path", "").split(" > ") if c and c.get("path") else []) if a]
    return anc + [label]

def vocab_ancestor_path(t, vocab):
    """Full ancestor path for a vocab type, formatted exactly like schema.org's `path` (top-down,
    ending at the immediate parent, EXCLUDING the type itself). Climbs parent links up through the
    vocabulary, then splices in the schema.org chain of the first schema ancestor it reaches. A
    `seen` guard keeps a bad re-parent (e.g. from consolidation) from looping forever."""
    up, cur, seen = [], t.get("parent"), set()
    while cur and cur not in seen:
        seen.add(cur)
        ent = _find_vocab_by_label(vocab, cur)
        if ent:                                   # another vocab type -> keep climbing the vocab tree
            up.append(cur); cur = ent.get("parent")
        else:                                     # a schema.org label (or Thing) -> splice its chain in
            up.extend(reversed(_schema_chain(cur))); cur = None
    return " > ".join(reversed(up))

def _annotate_vocab_paths(vocab):
    """Stamp each entry's derived `path` (Thing > ... > immediate parent), matching schema.org's `path`
    format. Derived from the `parent` chains (the source of truth) — refreshed on every save, so it
    stays correct even after consolidation re-parents a type."""
    for t in vocab["types"]:
        t["path"] = vocab_ancestor_path(t, vocab)
    return vocab

def _full_path(tref, vocab):
    """Full ancestor path for a resolved type reference (used in the review's chosen_type block):
    a vocab entry's parent chain, or a schema.org class's ancestor chain; None otherwise."""
    if tref.get("source") == "vocab" and tref.get("vocab_id"):
        ent = next((t for t in vocab["types"] if t["id"] == tref["vocab_id"]), None)
        if ent is not None:
            return vocab_ancestor_path(ent, vocab)
    if tref.get("source") == "schema.org" and tref.get("label"):
        return " > ".join(_schema_chain(tref["label"])[:-1])   # ancestors only (exclude the class itself)
    return None

def retrieve_candidates(type_texts, vocab):
    q_emb = embed(type_texts, QP)
    meta  = [("schema.org", c) for c in schema_pool]
    d_emb = schema_emb
    if vocab["types"]:
        v_emb = embed([vocab_embedding_text(t) for t in vocab["types"]], DP)
        d_emb = np.vstack([schema_emb, v_emb])
        meta  = meta + [("vocab", t) for t in vocab["types"]]
    sims = q_emb @ d_emb.T
    topk = np.argsort(-sims, axis=1)[:, :TOP_K_MATCH]
    out = []
    for ti in range(len(type_texts)):
        row = []
        for j in topk[ti]:
            src, obj = meta[j]
            if src == "schema.org":
                row.append({"source": "schema.org", "label": obj["label"], "path": obj.get("path", ""),
                            "score": round(float(sims[ti, j]), 3),
                            "comment": obj.get("enriched_comment") or obj.get("comment", "")})
            else:
                row.append({"source": "vocab", "id": obj["id"], "label": obj["canonical_label"],
                            "path": vocab_ancestor_path(obj, vocab), "score": round(float(sims[ti, j]), 3),
                            "comment": obj.get("definition", "")})
        out.append(row)
    return out

In [ ]:
# Article inputs + Gemini client (JSON mode, temperature 0 for reproducibility).
def load_article(aid):
    """Read the e2e single-call graph and group entities by their free-form `type`. e2e carries no per-type
    definitions/synonyms, so the embedding query is the type label + a few example mentions/aliases."""
    g = json.loads((RESULTS_DIR / aid / f"e2e/nary_graph{_sfx}.json").read_text())
    tinfo = {d["type"]: d for d in g.get("type_definitions", [])}   # nb2-style defs+synonyms (present once 6c is re-run)
    types = list(dict.fromkeys(e["type"] for e in g["entities"]))
    groups = {}
    for e in g["entities"]:
        groups.setdefault(e["type"], []).append(e)
    def qtext(t):
        info = tinfo.get(t, {})
        q = f'{t}: {info["definition"]}' if info.get("definition") else t
        if ALIGN_QUERY_STYLE and info.get("synonyms"):
            q = f'{q} Also called {", ".join(info["synonyms"])}.'
        ex = [e.get("text") for e in groups[t]] + [a for e in groups[t] for a in (e.get("aliases") or [])]
        ex = [x for x in dict.fromkeys(ex) if x][:4]
        if ALIGN_QUERY_STYLE and ex:
            q = f'{q} Examples: {", ".join(ex)}.'
        return q
    return g, tinfo, types, [qtext(t) for t in types], groups

from google import genai
from google.genai import types as genai_types
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
assert GEMINI_API_KEY, "Set GEMINI_API_KEY (or GOOGLE_API_KEY) in the env and restart the kernel."
client = genai.Client(api_key=GEMINI_API_KEY)

def gemini_json(system, user):
    r = client.models.generate_content(
        model=GEMINI_MODEL, contents=user,
        config=genai_types.GenerateContentConfig(
            system_instruction=system, temperature=0, response_mime_type="application/json"))
    return r.text

def parse_json(text):
    s = (text or "").strip()
    if s.startswith("```"):
        s = re.sub(r"^```[a-zA-Z]*\s*", "", s); s = re.sub(r"\s*```$", "", s)
    return json.loads(s)

In [ ]:
# ==== EDIT ME — prompt ===================================================================
# TYPE_DEDUP_PROMPT mirrors nb3's combined TASK 1 (type selection) + TASK 2 (within-group dedup),
# with two deltas: candidates may be Schema.org OR vocabulary (a "source" is reported), and the mint
# "parent" bullet lets a [vocab] candidate be the parent (nb3 is Schema.org-only). nb4 is
# self-contained (does its own dedup), so nb3 is NOT required.
TYPE_DEDUP_PROMPT = """You are an expert knowledge-engineering assistant. You are given several groups of entity mentions from one document. Each group shares a single free-form OIE type. For each group independently, perform two tasks, in order, and report them as separate fields (chosen_type / duplicate_clusters) — do not blend the two tasks, and NEVER merge or compare mentions across different groups.

Each group provides:
- OIE_TYPE: the free-form type label these mentions were given, plus its definition.
- CANDIDATES: a ranked list of candidate types proposed as matches for OIE_TYPE. Each candidate is either an existing Schema.org class or an existing type from our extended vocabulary; its source is marked [schema.org] or [vocab]. A candidate's `path` is its chain of ancestor classes — reported the same way for both sources — so every type you might use as a parent — a candidate label ([schema.org] or [vocab]) or any ancestor in a candidate's `path` (whether [schema.org] or [vocab]) — is already listed here.
- MENTIONS: the verbatim surface strings that were all assigned OIE_TYPE.

TASK 1 — TYPE SELECTION  (output field: "chosen_type")
Choose the single best type for the group.
- Prefer an existing candidate (Schema.org or vocabulary). Pick the one whose meaning (judged from its comment and path) best fits OIE_TYPE and the mentions. A candidate that is somewhat broader or narrower but still preserves what makes the type distinctive is preferable to a new type.
- Instance test — apply before accepting any candidate, judging by the MENTIONS, not by the OIE_TYPE definition alone: would each mention individually be a valid instance of that candidate — i.e. would it sit naturally alongside the concrete examples listed in the candidate's comment? If the mentions are open-ended categories, roles, or kinds rather than specific instances of the candidate, the candidate is too broad; reject it.
- Mint a new type when no candidate passes the instance test — i.e. no candidate genuinely fits, including when the best candidate is only "correct" by being so broad that it loses the type's distinguishing meaning. Do not settle for an over-broad candidate; instead mint a concise, reusable PascalCase label and a one-sentence definition that captures the type's specific meaning WITHOUT embedding a specific named entity/instance (use the general class, never a label containing a specific proper name); keep genuine class specificity (e.g. "TennisPlayer" is fine) but strip instance-specifics so the type applies to future mentions.
- When you mint, also assign a parent: the closest existing type the minted type should be a subclass of, so it slots into the hierarchy. The parent may be a Schema.org class OR a vocabulary [vocab] candidate type — prefer the most specific suitable parent already shown for this group (a candidate label, whether [schema.org] or [vocab], or any ancestor in a candidate's `path`, whether [schema.org] or [vocab]). A broader [vocab] candidate that the minted type is a kind of makes an ideal parent. Use another real Schema.org class only if none of the candidates fit, and fall back to "Thing" as a last resort. Parent is required for minted types and null otherwise.
- Report: source ("schema.org" or "vocab" when you chose a candidate, matching that candidate's marked source; "new" when you mint), the chosen label (the exact candidate label, or the new PascalCase label), whether it is new, its definition and parent (both only when new; else null), and a one-line justification.

TASK 2 — WITHIN-GROUP DEDUPLICATION  (output field: "duplicate_clusters")
Cluster this group's mentions that refer to the same real-world entity. Only ever cluster mentions that belong to the same group.
- Be conservative. Merge two mentions ONLY when they clearly denote the same specific entity, such as a short form or abbreviation and its full form. When unsure, keep them separate. Under-merging is much better than over-merging.
- NEVER merge overlapping-but-distinct entities: mentions that overlap in surface form or reference but denote different real-world entities must stay separate.
- For each cluster pick the canonical mention: the most complete / most explicit surface form. The canonical and every alias must be exact strings copied from that group's mentions.
- Output only clusters of size >= 2. Do not list singletons.

Return ONLY valid JSON — no prose, no code fences. Output one result object per input group, in the same order, each echoing its exact oie_type string. Angle-bracket placeholders show the shape, not literal values:
{"results": [{"oie_type": "<oie type>", "chosen_type": {"source": "schema.org | vocab | new", "label": "<type label>", "is_new": false, "definition": null, "parent": null, "justification": "one short line"}, "duplicate_clusters": [{"canonical": "<canonical mention>", "aliases": ["<alias mention>"]}]}]}
Include every group exactly once. If a group has nothing to merge, use "duplicate_clusters": []."""
print("prompt loaded")

prompt loaded


In [ ]:
# Hook A caller — ONE Gemini call per article: type selection + within-group dedup (both tasks).
def llm_select_and_dedup(types, tinfo, groups, cands):
    blocks = []
    for i, t in enumerate(types):
        b = [f"GROUP {i+1} —", f'OIE_TYPE: "{t}"']
        if tinfo.get(t, {}).get("definition"):
            b.append(f'OIE_TYPE_DEFINITION: {tinfo[t]["definition"]}')
        b.append("CANDIDATES (ranked):")
        if cands[i]:
            for k, c in enumerate(cands[i], 1):
                path = c.get("path") or "(datatype)"
                b.append(f'  {k}. [{c["source"]}] {c["label"]} | path/parent: {path}')
                b.append(f'     {c.get("comment", "")}')
        else:
            b.append("  (none)")
        b.append("MENTIONS:")
        b += [f'  - "{e["text"]}"' for e in groups[t]]
        blocks.append("\n".join(b))
    user = f"You are given {len(types)} GROUPS from one document.\n\n" + "\n\n".join(blocks)
    parsed  = parse_json(gemini_json(TYPE_DEDUP_PROMPT, user))
    results = parsed.get("results", parsed) if isinstance(parsed, dict) else parsed
    out = {r.get("oie_type"): {"chosen_type": r.get("chosen_type") or {},
                               "clusters": r.get("duplicate_clusters") or []}
           for r in (results or []) if r.get("oie_type") in set(types)}
    for i, t in enumerate(types):                    # fallback: omitted group -> top candidate, no dedup
        if t not in out:
            top = cands[i][0] if cands[i] else {"source": "schema.org", "label": "Thing"}
            out[t] = {"chosen_type": {"source": top["source"], "label": top["label"], "is_new": False,
                                      "justification": "fallback: top candidate"}, "clusters": []}
    return out

# ---- Minting: admit a chosen "new" type into the vocabulary -----------------------------------
# No mint-time canonicalization — the mint is admitted as-is. Collapsing synonymous / broader mints
# onto one entry is handled later by the OPTIONAL global consolidate_vocab pass.
def _schema_ref(label):
    return {"label": label, "source": "schema.org", "vocab_id": None,
            "definition": None, "parent": None, "is_new_to_vocab": False}

def _vocab_ref(ent, is_new=False):
    return {"label": ent["canonical_label"], "source": "vocab", "vocab_id": ent["id"],
            "definition": ent["definition"], "parent": ent["parent"], "is_new_to_vocab": is_new}

# Admit a minted type as a NEW vocabulary entry (or, in restrict mode, fall back to its parent).
def _apply_mint(aid, oie_type, ch, group, vocab):
    label = ch.get("label"); definition = ch.get("definition") or ""
    existing = _find_vocab_by_label(vocab, label)          # dedup by label: never mint a duplicate entry
    if existing is not None:                               # (the LLM may say is_new for a label already in vocab)
        _bump(existing, aid, oie_type, group)
        return _vocab_ref(existing)
    parent = ch.get("parent") or "Thing"
    if not EXPAND_MODE:                                        # restrict mode: fall back to the parent
        return _schema_ref(parent)
    if not (parent in SCHEMA_LABELS or _is_vocab_label(vocab, parent)):
        parent = "Thing"
    ent = {"id": f"T{vocab['next_id']:04d}", "canonical_label": label, "definition": definition,
           "parent": parent, "aliases": [], "support": 1, "first_seen": aid,
           "provenance": [{"article": aid, "oie_type": oie_type, "mentions": [e["text"] for e in group]}]}
    vocab["types"].append(ent); vocab["next_id"] += 1
    return _vocab_ref(ent, is_new=True)

# Entity-node construction (own dedup, ported from nb3 — every mention lands in exactly one node).
_META_KEYS = ("text", "normalized_value", "normalized_end", "confidence")

def _make_node(oie_type, tref, canonical, alias_texts, by_text):
    meta = [by_text[m] for m in ([canonical] + alias_texts) if m in by_text]
    return {
        "canonical_text":         canonical,
        "aliases":                alias_texts,
        "oie_type":               oie_type,
        "chosen_type":            tref.get("label"),
        "type_source":            tref.get("source"),
        "vocab_id":               tref.get("vocab_id"),
        "type_is_new_to_vocab":   tref.get("is_new_to_vocab", False),
        "type_definition":        tref.get("definition"),
        "type_parent":            tref.get("parent"),
        "type_justification":     tref.get("justification"),
        "mention_count":          len(meta),
        "mentions":               [{k: m.get(k) for k in _META_KEYS} for m in meta],
    }

def _apply_clusters(oie_type, entities, tref, clusters):
    """LLM clusters -> canonical nodes; conservative (hallucinated/duplicate members are skipped, so
    every mention lands in exactly one node)."""
    by_text, used, nodes = {e["text"]: e for e in entities}, set(), []
    for cl in clusters or []:
        canonical = cl.get("canonical")
        if canonical not in by_text or canonical in used:
            continue
        aliases = [a for a in (cl.get("aliases") or [])
                   if a in by_text and a != canonical and a not in used]
        if not aliases:
            continue
        for m in [canonical] + aliases:
            used.add(m)
        nodes.append(_make_node(oie_type, tref, canonical, aliases, by_text))
    for e in entities:                                          # remaining mentions -> singletons
        if e["text"] not in used:
            used.add(e["text"])
            nodes.append(_make_node(oie_type, tref, e["text"], [], by_text))
    return nodes

In [ ]:
# Process ONE article: retrieve (Hook A) -> ONE LLM call (nb4's type-selection; we use only chosen_type,
# NOT the dedup clusters) -> admit minted types -> RELABEL entities BY ID (ids preserved so relations stay
# valid). Writes e2e/nary_graph_typed.json + e2e/entity_type_matching_review.json.
def process_article(aid, vocab, write=True, save=True, verbose=True):
    g, tinfo, types, qtexts, groups = load_article(aid)
    cands  = retrieve_candidates(qtexts, vocab)
    report = llm_select_and_dedup(types, tinfo, groups, cands)     # reuse nb4; only chosen_type is used

    resolved, n_mint = {}, 0
    for t in types:
        ch = report[t]["chosen_type"]
        is_mint = ch.get("is_new") or ch.get("source") == "new"
        if is_mint:
            before = len(vocab["types"])
            tref = _apply_mint(aid, t, ch, groups[t], vocab)
            n_mint += (len(vocab["types"]) > before)
        else:
            ent = _find_vocab_by_label(vocab, ch.get("label"))
            if ent:
                _bump(ent, aid, t, groups[t]); tref = _vocab_ref(ent)
            else:
                tref = _schema_ref(ch.get("label"))
        tref["justification"] = ch.get("justification")
        resolved[t] = tref

    # Relabel entities BY ID — preserve id/text/aliases so the graph's relations stay valid (NO merge).
    ents = []
    for e in g["entities"]:
        tref = resolved[e["type"]]
        ents.append({**e,
            "type":                 tref.get("label"),      # canonical label (schema.org or vocab)
            "type_oie":             e["type"],               # keep the original free-form type
            "type_source":          tref.get("source"),
            "type_vocab_id":        tref.get("vocab_id"),
            "type_is_new_to_vocab": tref.get("is_new_to_vocab", False),
            "type_definition":      tref.get("definition"),
            "type_parent":          tref.get("parent"),
            "type_justification":   tref.get("justification")})
    id2type = {e["id"]: e["type"] for e in ents}             # n-ary node ids stay out -> keep their cached type
    rels, _bad = [], 0
    for r in g.get("relations", []):
        if not r.get("subject_id") or not r.get("object_id"):   # drop malformed half-edges (missing an endpoint)
            _bad += 1; continue
        rels.append({**r,
                     "subject_type": id2type.get(r["subject_id"], r.get("subject_type")),
                     "object_type":  id2type.get(r["object_id"],  r.get("object_type"))})
    if _bad:
        print(f"    [warn] {aid}: dropped {_bad} relation(s) missing subject_id/object_id")

    if write:
        out_dir = RESULTS_DIR / aid / "e2e"
        (out_dir / f"nary_graph{_sfx}_typed.json").write_text(json.dumps(
            {**{k: g[k] for k in ("article_id", "reference_date", "method") if k in g},
             "match_model": MATCH_MODEL, "expand_mode": EXPAND_MODE,
             "entities": ents, "nary_nodes": g.get("nary_nodes", []),
             "relations": rels, "relation_definitions": g.get("relation_definitions", [])},
            indent=2, ensure_ascii=False))
        review = [{
            "oie_type":             t,
            "embedding_query_text": qtexts[i],
            "chosen_type": {
                "label":           resolved[t].get("label"),
                "source":          resolved[t].get("source"),
                "vocab_id":        resolved[t].get("vocab_id"),
                "is_new_to_vocab": resolved[t].get("is_new_to_vocab", False),
                "definition":      resolved[t].get("definition"),
                "parent":          resolved[t].get("parent"),
                "path":            _full_path(resolved[t], vocab),
                "justification":   resolved[t].get("justification")},
            "llm_proposed":         {k: (report[t]["chosen_type"] or {}).get(k) for k in ("source", "label", "is_new")},
            "mentions":             [e["text"] for e in groups[t]],
            "candidates":           cands[i],
        } for i, t in enumerate(types)]
        (out_dir / f"entity_type_matching_review{_sfx}.json").write_text(json.dumps(
            {"article_id": aid, "match_model": MATCH_MODEL, "expand_mode": EXPAND_MODE, "types": review},
            indent=2, ensure_ascii=False))
    if save:
        save_vocab(vocab)
    if verbose:
        nvoc = sum(1 for e in ents if e.get("type_source") == "vocab")
        print(f"  {aid}: {len(types)} distinct types over {len(ents)} entities "
              f"| +{n_mint} minted | {nvoc} via vocab | vocab now {len(vocab['types'])}")
    return ents, vocab

In [ ]:
# ── MERGED type canonicalization: ONE Gemini call per article for BOTH entity types AND Event-hub types.
# Replaces the old two-pass approach (entity Build + a separate Event-hub pass = 2 calls/article) with a
# single call/article (~50% fewer calls). Each group carries its OWN retrieved CANDIDATES, so entity
# types still match the FULL schema.org pool while Event types match only the Event subtree; the "every
# Event type is a subtype of Event" guarantee is enforced at retrieval + mint (per KIND), NOT by the calls
# being separate. Both kinds grow the SAME _global/e2e/type_vocabulary.json.
# Selection-only: the old within-group dedup (TASK 2) is dropped — the single 6c call already resolved
# coreference and 6d never used those clusters (it relabels entities by id, no merge).

EVENT_ROOT = "Event"

def _under_event_schema(label):
    return EVENT_ROOT in _schema_chain(label)                       # _schema_chain includes `label`

def _under_event_vocab(t, vocab):
    return t["canonical_label"] == EVENT_ROOT or \
           EVENT_ROOT in vocab_ancestor_path(t, vocab).split(" > ")

def retrieve_event_candidates(type_texts, vocab):
    """Top-k retrieval like retrieve_candidates, but the pool is restricted to the Event subtree
    (schema.org Event-subtree classes u vocab types parented under Event)."""
    sch  = [c for c in schema_pool if _under_event_schema(c["label"])]
    meta = [("schema.org", c) for c in sch]
    docs = [c["embedding_text"] for c in sch]
    for t in vocab["types"]:
        if _under_event_vocab(t, vocab):
            meta.append(("vocab", t)); docs.append(vocab_embedding_text(t))
    if not docs:
        return [[] for _ in type_texts]
    d_emb = embed(docs, DP); q_emb = embed(type_texts, QP)
    sims  = q_emb @ d_emb.T
    topk  = np.argsort(-sims, axis=1)[:, :TOP_K_MATCH]
    out = []
    for ti in range(len(type_texts)):
        row = []
        for j in topk[ti]:
            src, obj = meta[j]
            if src == "schema.org":
                row.append({"source": "schema.org", "label": obj["label"], "path": obj.get("path", ""),
                            "score": round(float(sims[ti, j]), 3),
                            "comment": obj.get("enriched_comment") or obj.get("comment", "")})
            else:
                row.append({"source": "vocab", "id": obj["id"], "label": obj["canonical_label"],
                            "path": vocab_ancestor_path(obj, vocab), "score": round(float(sims[ti, j]), 3),
                            "comment": obj.get("definition", "")})
        out.append(row)
    return out

def _bump_hub(ent, aid, etype, group):
    ent["support"] = ent.get("support", 0) + 1
    ent.setdefault("provenance", []).append(
        {"article": aid, "oie_type": etype, "mentions": [h.get("label") for h in group]})

def _apply_mint_hub(aid, etype, ch, group, vocab):
    """Admit a minted Event type; parent is forced into the Event subtree (default 'Event', never 'Thing')."""
    parent, label = ch.get("parent") or EVENT_ROOT, ch.get("label")
    existing = _find_vocab_by_label(vocab, label)          # dedup by label: never mint a duplicate entry
    if existing is not None:
        _bump_hub(existing, aid, etype, group)
        return _vocab_ref(existing)
    ok = (parent in SCHEMA_LABELS and _under_event_schema(parent))
    if not ok:
        pe = _find_vocab_by_label(vocab, parent)
        ok = bool(pe and _under_event_vocab(pe, vocab))
    if not ok:
        parent = EVENT_ROOT
    if not EXPAND_MODE:
        return _schema_ref(parent)
    ent = {"id": f"T{vocab['next_id']:04d}", "canonical_label": label, "definition": ch.get("definition") or "",
           "parent": parent, "aliases": [], "support": 1, "first_seen": aid,
           "provenance": [{"article": aid, "oie_type": etype, "mentions": [h.get("label") for h in group]}]}
    vocab["types"].append(ent); vocab["next_id"] += 1
    return _vocab_ref(ent, is_new=True)

# ==== EDIT ME — unified selection prompt (entity + event groups in one call, selection only) =========
TYPE_SELECT_PROMPT = """You are an expert knowledge-engineering assistant. You are given several GROUPS from one news document. Each group is a single free-form OIE type to canonicalize, and is one of two KINDS:
- ENTITY — a type for real-world entities (people, organizations, places, quantities, ...). Its chosen type may be ANY schema.org class or vocabulary type.
- EVENT — a type for a KIND of event. Its chosen type MUST be a subtype of schema.org Event.

Each group provides:
- KIND: ENTITY or EVENT.
- OIE_TYPE: the free-form type label (plus its definition, if any).
- CANDIDATES: a ranked list of existing types proposed as matches — each an existing Schema.org class or an existing type from our extended vocabulary, marked [schema.org] or [vocab], with its ancestor `path` (so every label or ancestor shown is available as a parent).
- MENTIONS / EXAMPLES: the surface strings (ENTITY) or event titles (EVENT) that were given this type.

For each group independently — NEVER compare or merge across groups — choose the single best type:
- Prefer reusing an existing candidate (Schema.org or [vocab]) whose meaning, judged from its comment and path, best fits OIE_TYPE and the mentions/examples. A candidate that is somewhat broader or narrower but still preserves what makes the type distinctive beats minting a new one.
- Instance test — judge by the MENTIONS/EXAMPLES, not the definition alone: would each individually be a valid instance of the candidate? If the mentions are open-ended categories, roles, or kinds rather than instances of the candidate, it is too broad — reject it.
- Mint a new type only when no candidate passes the instance test. Give a concise, reusable PascalCase label with NO specific named entity in it, a one-sentence definition, and a parent: the closest existing type it is a subclass of (a candidate label, or any ancestor in a candidate's `path`, whether [schema.org] or [vocab]).
- CONSTRAINT for KIND = EVENT: the chosen type (reused OR minted) MUST be a subtype of schema.org Event, and a minted Event type's parent MUST be schema.org "Event", a more specific Event-subtree class, or a [vocab] event type shown in CANDIDATES — never "Thing". For KIND = ENTITY, use "Thing" only as a last resort.
- Report per group: source ("schema.org" | "vocab" when reusing a candidate, matching that candidate's marked source; "new" when you mint), label (the exact candidate label, or the new PascalCase label), is_new (bool), definition + parent (both only when new; else null), and a one-line justification.

Return ONLY valid JSON — no prose, no code fences. Output one object per input group, in the same order, each echoing its exact GROUP id:
{"results": [{"id": "<group id>", "chosen_type": {"source": "schema.org | vocab | new", "label": "<type label>", "is_new": false, "definition": null, "parent": null, "justification": "one short line"}}]}
Include every group exactly once."""

def llm_select_types(items):
    """ONE Gemini call over ENTITY + EVENT groups. `items`: dicts with gid/kind/oie_type/definition/
    examples/candidates. Returns {gid: chosen_type}. Missing groups fall back to the top candidate."""
    blocks = []
    for it in items:
        b = [f'GROUP {it["gid"]} — KIND: {it["kind"].upper()}', f'OIE_TYPE: "{it["oie_type"]}"']
        if it.get("definition"):
            b.append(f'DEFINITION: {it["definition"]}')
        b.append("CANDIDATES (ranked):")
        if it["candidates"]:
            for k, c in enumerate(it["candidates"], 1):
                b.append(f'  {k}. [{c["source"]}] {c["label"]} | path/parent: {c.get("path") or "(datatype)"}')
                b.append(f'     {c.get("comment", "")}')
        else:
            b.append("  (none)")
        b.append("EXAMPLES:" if it["kind"] == "event" else "MENTIONS:")
        b += [f'  - "{x}"' for x in (it["examples"] or [])[:6]]
        blocks.append("\n".join(b))
    user = f"You are given {len(items)} GROUPS from one document.\n\n" + "\n\n".join(blocks)
    parsed  = parse_json(gemini_json(TYPE_SELECT_PROMPT, user))
    results = parsed.get("results", parsed) if isinstance(parsed, dict) else parsed
    out = {r.get("id"): (r.get("chosen_type") or {}) for r in (results or []) if r.get("id")}
    for it in items:                                          # fallback: omitted group -> top candidate
        if it["gid"] not in out:
            top = it["candidates"][0] if it["candidates"] else \
                  {"source": "schema.org", "label": EVENT_ROOT if it["kind"] == "event" else "Thing"}
            out[it["gid"]] = {"source": top["source"], "label": top["label"], "is_new": False,
                              "justification": "fallback: top candidate"}
    return out

def canonicalize_article_types(aid, vocab, write=True, save=True, verbose=True):
    """ONE LLM call canonicalizes entity types (full schema.org pool) AND Event-hub types (Event subtree)
    for one article. Relabels entities + Event hubs BY ID (relations stay valid), grows `vocab`, and writes
    nary_graph{_sfx}_typed.json + entity/hub review files. Observation/Claim hub types stay free-form."""
    g = json.loads((RESULTS_DIR / aid / f"e2e/nary_graph{_sfx}.json").read_text())
    tinfo = {d["type"]: d for d in g.get("type_definitions", [])}

    # ENTITY groups — query text mirrors load_article's qtext ------------------------------------
    ent_types  = list(dict.fromkeys(e["type"] for e in g["entities"]))
    ent_groups = {t: [e for e in g["entities"] if e["type"] == t] for t in ent_types}
    def _eq(t):
        info = tinfo.get(t, {})
        q = f'{t}: {info["definition"]}' if info.get("definition") else t
        if ALIGN_QUERY_STYLE and info.get("synonyms"):
            q = f'{q} Also called {", ".join(info["synonyms"])}.'
        ex = [e.get("text") for e in ent_groups[t]] + [a for e in ent_groups[t] for a in (e.get("aliases") or [])]
        ex = [x for x in dict.fromkeys(ex) if x][:4]
        if ALIGN_QUERY_STYLE and ex:
            q = f'{q} Examples: {", ".join(ex)}.'
        return q
    ent_q     = [_eq(t) for t in ent_types]
    ent_cands = retrieve_candidates(ent_q, vocab)

    # EVENT hub groups (node_kind == "Event"; their `type` is still the 6c free-form label) -------
    ev          = [n for n in g.get("nary_nodes", []) if n.get("node_kind") == EVENT_ROOT]
    ev_types    = list(dict.fromkeys(n.get("type") for n in ev))
    ev_groups   = {t: [n for n in ev if n.get("type") == t] for t in ev_types}
    ev_examples = {t: [n.get("label", "") for n in ev_groups[t]] for t in ev_types}
    ev_q = []
    for t in ev_types:
        ex = ", ".join(x for x in ev_examples[t][:4] if x)
        ev_q.append(f'{t} (examples: {ex})' if ex else t)
    ev_cands = retrieve_event_candidates(ev_q, vocab)

    # ONE combined LLM call ---------------------------------------------------------------------
    items  = [{"gid": f"E{i+1}", "kind": "entity", "oie_type": t, "definition": tinfo.get(t, {}).get("definition"),
               "examples": [e["text"] for e in ent_groups[t]], "candidates": ent_cands[i]}
              for i, t in enumerate(ent_types)]
    items += [{"gid": f"V{i+1}", "kind": "event", "oie_type": t, "definition": None,
               "examples": ev_examples[t], "candidates": ev_cands[i]}
              for i, t in enumerate(ev_types)]
    report = llm_select_types(items) if items else {}

    # Resolve ENTITY types (mint parents anywhere; Thing fallback) -------------------------------
    ent_res, n_mint_e = {}, 0
    for i, t in enumerate(ent_types):
        ch = report.get(f"E{i+1}", {})
        if ch.get("is_new") or ch.get("source") == "new":
            before = len(vocab["types"]); tref = _apply_mint(aid, t, ch, ent_groups[t], vocab)
            n_mint_e += (len(vocab["types"]) > before)
        else:
            ent = _find_vocab_by_label(vocab, ch.get("label"))
            if ent: _bump(ent, aid, t, ent_groups[t]); tref = _vocab_ref(ent)
            else:   tref = _schema_ref(ch.get("label"))
        tref["justification"] = ch.get("justification"); ent_res[t] = tref

    # Resolve EVENT types (mint parents forced under Event) --------------------------------------
    ev_res, n_mint_v = {}, 0
    for i, t in enumerate(ev_types):
        ch = report.get(f"V{i+1}", {})
        if ch.get("is_new") or ch.get("source") == "new":
            before = len(vocab["types"]); tref = _apply_mint_hub(aid, t, ch, ev_groups[t], vocab)
            n_mint_v += (len(vocab["types"]) > before)
        else:
            ent = _find_vocab_by_label(vocab, ch.get("label"))
            if ent: _bump_hub(ent, aid, t, ev_groups[t]); tref = _vocab_ref(ent)
            else:   tref = _schema_ref(ch.get("label"))
        tref["justification"] = ch.get("justification"); ev_res[t] = tref

    # Relabel entities BY ID (preserve id/text/aliases; NO merge) --------------------------------
    ents = []
    for e in g["entities"]:
        tref = ent_res[e["type"]]
        ents.append({**e, "type": tref.get("label"), "type_oie": e["type"],
                     "type_source": tref.get("source"), "type_vocab_id": tref.get("vocab_id"),
                     "type_is_new_to_vocab": tref.get("is_new_to_vocab", False),
                     "type_definition": tref.get("definition"), "type_parent": tref.get("parent"),
                     "type_justification": tref.get("justification")})

    # Relabel Event hubs BY ID (keep type_oie); leave Observation/Claim types free-form ----------
    nary = [dict(n) for n in g.get("nary_nodes", [])]
    for n in nary:
        if n.get("node_kind") == EVENT_ROOT:
            t = n.get("type"); tref = ev_res[t]
            n.update({"type": tref.get("label"), "type_oie": t, "type_source": tref.get("source"),
                      "type_vocab_id": tref.get("vocab_id"),
                      "type_is_new_to_vocab": tref.get("is_new_to_vocab", False),
                      "type_definition": tref.get("definition"), "type_parent": tref.get("parent"),
                      "type_justification": tref.get("justification")})

    # Refresh relation endpoint types from entities + all n-ary nodes ----------------------------
    id2type = {e["id"]: e["type"] for e in ents}
    id2type.update({n["id"]: n.get("type") for n in nary})
    rels, _bad = [], 0
    for r in g.get("relations", []):
        if not r.get("subject_id") or not r.get("object_id"):    # drop malformed half-edges
            _bad += 1; continue
        rels.append({**r, "subject_type": id2type.get(r["subject_id"], r.get("subject_type")),
                          "object_type":  id2type.get(r["object_id"],  r.get("object_type"))})
    if _bad:
        print(f"    [warn] {aid}: dropped {_bad} relation(s) missing subject_id/object_id")

    if write:
        out_dir = RESULTS_DIR / aid / "e2e"
        (out_dir / f"nary_graph{_sfx}_typed.json").write_text(json.dumps(
            {**{k: g[k] for k in ("article_id", "reference_date", "method") if k in g},
             "match_model": MATCH_MODEL, "expand_mode": EXPAND_MODE,
             "entities": ents, "nary_nodes": nary,
             "relations": rels, "relation_definitions": g.get("relation_definitions", [])},
            indent=2, ensure_ascii=False))
        ent_review = [{
            "oie_type": t, "embedding_query_text": ent_q[i],
            "chosen_type": {"label": ent_res[t].get("label"), "source": ent_res[t].get("source"),
                "vocab_id": ent_res[t].get("vocab_id"), "is_new_to_vocab": ent_res[t].get("is_new_to_vocab", False),
                "definition": ent_res[t].get("definition"), "parent": ent_res[t].get("parent"),
                "path": _full_path(ent_res[t], vocab), "justification": ent_res[t].get("justification")},
            "llm_proposed": {k: (report.get(f"E{i+1}", {}) or {}).get(k) for k in ("source", "label", "is_new")},
            "mentions": [e["text"] for e in ent_groups[t]], "candidates": ent_cands[i]}
            for i, t in enumerate(ent_types)]
        (out_dir / f"entity_type_matching_review{_sfx}.json").write_text(json.dumps(
            {"article_id": aid, "match_model": MATCH_MODEL, "expand_mode": EXPAND_MODE, "types": ent_review},
            indent=2, ensure_ascii=False))
        (out_dir / f"hub_type_matching_review{_sfx}.json").write_text(json.dumps(
            {"article_id": aid, "types": [{"event_type": t, "chosen": ev_res[t],
              "examples": ev_examples[t], "candidates": ev_cands[i]} for i, t in enumerate(ev_types)]},
            indent=2, ensure_ascii=False))
    if save:
        save_vocab(vocab)
    if verbose:
        nvoc = sum(1 for e in ents if e.get("type_source") == "vocab")
        print(f"  {aid}: {len(ent_types)} entity + {len(ev_types)} event types over {len(ents)} entities "
              f"| +{n_mint_e + n_mint_v} minted | {nvoc} ent via vocab | vocab now {len(vocab['types'])}")
    return ents, nary, vocab


## Build — incremental vocabulary over all articles (in order)

In [ ]:
import time
# Incremental build — ONE Gemini call per article canonicalizes BOTH entity types AND Event-hub types
# (merged pass; ~50% fewer calls than the old entity-Build + separate Event-hub pass).
# FRESH=True: start from an EMPTY e2e vocab and process ALL of ARTICLE_ORDER.
# FRESH=False: keep the existing e2e type_vocabulary.json and process only NEW_ARTICLES.
FRESH        = True
NEW_ARTICLES = []

if FRESH:
    vocab = {"types": [], "next_id": 1}; save_vocab(vocab); todo = ARTICLE_ORDER
else:
    vocab = load_vocab()
    todo  = NEW_ARTICLES or [a for a in ARTICLE_ORDER
                             if not (RESULTS_DIR / a / f"e2e/nary_graph{_sfx}_typed.json").exists()]

print(f"building e2e type vocabulary over {len(todo)} article(s): {todo}\n")
for aid in todo:
    canonicalize_article_types(aid, vocab, write=True, save=True, verbose=True)
    time.sleep(5)
print(f"\ndone — {len(vocab['types'])} canonical types in {VOCAB_PATH}")


building e2e type vocabulary over 4 article(s): ['art1', 'art2', 'art4', 'art5']



KeyboardInterrupt: 

## Event-hub types — folded into the Build above (no separate call)

Event n-ary node types are now canonicalized in the **same per-article call** as entity types (the merged
`canonicalize_article_types` pass), with each Event group's candidates restricted to the schema.org **Event
subtree** so every result stays a subtype of `Event`. This halves the LLM calls in 6d (was: entity Build +
a separate Event-hub pass = 2 calls/article; now: 1 call/article). Observation/Claim hub types are left
free-form. The old standalone Event-hub cell has been removed.


## Report + verify

In [ ]:
# e2e type-vocabulary summary, then verify each typed graph preserves its input entity ids.
vocab = load_vocab()
print(f"E2E TYPE VOCABULARY — {len(vocab['types'])} canonical types\n" + "-" * 74)
for t in sorted(vocab["types"], key=lambda x: -x.get("support", 0)):
    print(f"  {t['id']}  {t['canonical_label']:<24} <: {str(t.get('parent')):<16} support={t.get('support')}")

print("\nVERIFY typed graph vs e2e/nary_graph.json (entity ids preserved)\n" + "-" * 74)
ok = True
for aid in ARTICLE_ORDER:
    tf = RESULTS_DIR / aid / f"e2e/nary_graph{_sfx}_typed.json"
    if not tf.exists():
        print(f"  {aid:<8} (no typed file)"); continue
    raw = json.loads((RESULTS_DIR / aid / f"e2e/nary_graph{_sfx}.json").read_text())
    typ = json.loads(tf.read_text())
    ids_raw = sorted(e["id"] for e in raw["entities"])
    ids_typ = sorted(e["id"] for e in typ["entities"])
    nvoc = sum(1 for e in typ["entities"] if e.get("type_source") == "vocab")
    same = ids_raw == ids_typ; ok &= same
    print(f"  {aid:<8} {len(ids_raw):>4} entities  ids_preserved={same}  ({nvoc} typed via vocab)")
print("-" * 74); print("consistent" if ok else "INCONSISTENCIES")

E2E TYPE VOCABULARY — 92 canonical types
--------------------------------------------------------------------------
  T0023  AlertIssuedEvent         <: Event            support=3
  T0001  Month                    <: Intangible       support=2
  T0002  DateRange                <: Intangible       support=2
  T0003  Temperature              <: Quantity         support=2
  T0006  HolidayWeekend           <: DateRange        support=2
  T0012  AgeGroup                 <: PeopleAudience   support=2
  T0038  RepresentativeGroup      <: Thing            support=2
  T0056  Year                     <: Thing            support=2
  T0067  CeremonyStartEvent       <: Event            support=2
  T0004  AlertLevel               <: StatusEnumeration support=1
  T0005  Archipelago              <: Landform         support=1
  T0007  Phenomenon               <: Intangible       support=1
  T0008  WeatherEvent             <: Event            support=1
  T0009  Resource                 <: Thing         

# ─────────────────────────── GLOBAL RE-CONSOLIDATION (optional) ───────────────────────────
## Everything below is the optional global-reconsolidation block

Run the pipeline above (through **Build** and **Report + verify**) to produce the raw incremental
vocabulary (`type_vocabulary.json`). The cells below are a **self-contained, optional block** — run
them to clean up the vocabulary globally, or **skip them entirely** to use the raw vocabulary as-is.
Nothing above depends on these cells, and **this block writes only NEW files** — every incremental
output (`type_vocabulary.json`, the per-article `*_ext.json`) is left completely untouched, so you can
always compare before/after.


The one caveat is scale. This works only while the whole vocabulary (the extension only) fits comfortably in a single prompt. If the vocab grows into the many hundreds/thousands, you'd have to reintroduce embeddings — either to chunk the list or to shortlist candidate pairs — so the model isn't asked to reason over an unwieldy list in one shot. 


The block, in order (all outputs are separate from the incremental build):
1. **Config + prompt** — `SUBTYPE_MODE`, the separate output paths, `load_consolidated`, `CONSOLIDATE_ALL_PROMPT`.
2. **`consolidate_vocab`** — ONE Gemini call → writes `e2e type_vocabulary_consolidated.json` + `type_consolidation_report.json` (raw `type_vocabulary.json` untouched).
3. **`refresh_ext_after_consolidation`** — no LLM; writes a separate `nary_graph_typed_consolidated.json` per article (incremental `*_ext.json` untouched) **plus** a changes-only file `consolidation_ext_changes.json` (before → after of every node that moved).
4. **`build_type_hierarchy`** — indented schema.org + extension tree, written to TWO separate files: `type_hierarchy.txt` (raw incremental) and `type_hierarchy_consolidated.txt` (consolidated).

In [ ]:
# Global consolidation prompt — used by the ONE-call consolidate_vocab below.
# SUBTYPE_MODE controls how a SPECIFIC type is related to a broader type ALREADY in the vocabulary:
#   "off"      : only merge exact synonyms (no broader/narrower handling).
#   "subclass" : KEEP the specific type, set its parent to the broader vocab type — builds an is-a
#                hierarchy; preserves granularity + is-a links.
#   "merge"    : COLLAPSE the specific type into the broader parent (child removed, folded into parent).
SUBTYPE_MODE = "subclass"
assert SUBTYPE_MODE in ("off", "subclass", "merge"), "SUBTYPE_MODE must be 'off', 'subclass', or 'merge'"
CONSOLIDATE_TOP_K = 5   # retrieve-then-judge: nearest-neighbour shortlist size per item

# Consolidation writes to SEPARATE files — the raw incremental type_vocabulary.json is left INTACT.
CONSOLIDATED_VOCAB_PATH   = RESULTS_DIR / f"_global/e2e/type_vocabulary{_sfx}_consolidated.json"   # the consolidated snapshot
CONSOLIDATION_REPORT_PATH = RESULTS_DIR / f"_global/e2e/type_consolidation_report{_sfx}.json"      # what merged / re-parented
def load_consolidated():
    """The consolidated vocab if it exists, else fall back to the raw incremental one."""
    return json.loads(CONSOLIDATED_VOCAB_PATH.read_text()) if CONSOLIDATED_VOCAB_PATH.exists() else load_vocab()

CONSOLIDATE_ALL_PROMPT = """You are organizing a vocabulary of entity TYPES. You are given the full list of types (id, label, parent, support, definition). Produce GROUPS of two kinds:

SYNONYM — two or more entries that denote the SAME concept (true synonyms: the same kind of thing, differing only in wording or label). Set keep_id to any member and merge_ids to the rest.
SUBTYPE — one or more SPECIFIC (child) types that are all kinds of a single more GENERAL (parent) type WHICH IS ALSO PRESENT in this vocabulary — an is-a relationship. Set keep_id to the GENERAL parent type's id and merge_ids to the specific child ones. Only do this when the parent type already appears in the list (never invent one) and genuinely subsumes the children — every mention under a child type must still be a valid instance of the parent type.

Rules:
- Judge by each type's LABEL and DEFINITION (not the label alone). Be conservative. For SYNONYM, only group true equivalents. For SUBTYPE, only relate a child to a parent when the parent clearly and safely subsumes it; if a real distinction would be lost, leave them separate.
- Each type belongs to at most one group. Every group has a keep_id and one or more merge_ids. Give a one-line reason and set "kind" to "synonym" or "subtype".

Return ONLY valid JSON — no prose, no code fences. Angle-bracket placeholders show the shape, not literal values:
{"groups": [{"kind": "subtype", "keep_id": "<parent id>", "merge_ids": ["<child id>"], "reason": "one short line"}, {"kind": "synonym", "keep_id": "<id>", "merge_ids": ["<id>"], "reason": "one short line"}]}"""
print(f"consolidation prompt loaded | SUBTYPE_MODE={SUBTYPE_MODE!r} | consolidated -> {CONSOLIDATED_VOCAB_PATH.name}")

consolidation prompt loaded | SUBTYPE_MODE='subclass' | consolidated -> type_vocabulary_dedup_consolidated.json


In [ ]:
import time

def consolidate_vocab(vocab, write_report=True, verbose=True):
    """Global re-consolidation in ONE Gemini call. The model returns SYNONYM and SUBTYPE groups;
    how SUBTYPE (is-a) groups are applied depends on SUBTYPE_MODE:
      "off"      -> ignore them (only synonyms merge).
      "subclass" -> KEEP each child type, set its parent to the broader vocab type (build hierarchy).
      "merge"    -> fold each child type into the broader parent (removed; label -> alias).
    SYNONYM groups always merge onto the highest-support member. Writes the consolidated vocab to
    CONSOLIDATED_VOCAB_PATH and a report to CONSOLIDATION_REPORT_PATH — the raw type_vocabulary.json is
    NOT modified (pass load_vocab() as `vocab`)."""
    ents  = list(vocab["types"])
    by_id = {t["id"]: t for t in ents}

    def _articles_of(t):
        return [p.get("article") for p in t.get("provenance", []) if p.get("article")]
    articles_before = sorted({a for t in ents for a in _articles_of(t)})

    groups = []
    shortlist, _adj = [], {}     # the embedder's proposals, for the audit report
    if len(ents) >= 2:
        ents_sorted = sorted(ents, key=lambda x: x["id"])
        lines = [f'- id: {t["id"]} | label: {t["canonical_label"]} | parent: {t.get("parent")} '
                 f'| support: {t.get("support", 0)} | def: {t.get("definition", "")}'
                 + (f' | aka: {", ".join(t["aliases"])}' if t.get("aliases") else "")
                 for t in ents_sorted]
        # Retrieve-then-judge: shortlist each type's TOP-K nearest neighbours by embedding
        # (against the full pool) so the LLM only weighs plausible pairs, not every possible pair.
        _ids = [t["id"] for t in ents_sorted]
        _lab = {t["id"]: t["canonical_label"] for t in ents_sorted}
        _E = embed([vocab_embedding_text(t) for t in ents_sorted], DP)
        _sims = _E @ _E.T
        np.fill_diagonal(_sims, -1.0)
        _k = min(CONSOLIDATE_TOP_K, len(ents_sorted) - 1)
        nbrs = []
        for i in range(len(ents_sorted)):
            top = np.argsort(-_sims[i])[:_k]
            _adj[_ids[i]] = {_ids[j] for j in top}                # what the embedder proposed
            shortlist.append({"id": _ids[i], "label": _lab[_ids[i]],
                              "neighbours": [{"id": _ids[j], "label": _lab[_ids[j]],
                                              "score": round(float(_sims[i][j]), 3)} for j in top]})
            nbrs.append(f'{_ids[i]}: ' + ", ".join(_ids[j] for j in top))
        gate = ("\nBoth SYNONYM and SUBTYPE groups are enabled." if SUBTYPE_MODE != "off" else
                "\nReturn only SYNONYM groups; do NOT relate broader/narrower (subtype) types.")
        gate += ("\nCANDIDATE_NEIGHBOURS lists, per type, its most similar candidates. Only group "
                 "types that appear together in some neighbour line; never propose a pair not listed there.")
        msg = "VOCABULARY:\n" + "\n".join(lines) + "\n\nCANDIDATE_NEIGHBOURS:\n" + "\n".join(nbrs)
        try:
            res = parse_json(gemini_json(CONSOLIDATE_ALL_PROMPT + gate, msg))
            groups = (res.get("groups", res) if isinstance(res, dict) else res) or []
        except Exception as ex:
            print(f"[WARN] consolidation call failed ({type(ex).__name__}: {ex})")

    merges, reparents, merged_ids = [], [], set()
    for g in groups:
        kind = (g.get("kind") or "synonym").lower()
        keep = g.get("keep_id")
        ids  = list(dict.fromkeys(i for i in ([keep] + (g.get("merge_ids") or []))
                                  if i in by_id and i not in merged_ids))
        if len(ids) < 2:
            continue
        is_subtype = kind.startswith("subtype")

        # SUBTYPE + subclass: keep the children, just re-parent them onto the broader parent
        if is_subtype and SUBTYPE_MODE == "subclass":
            if keep not in by_id or keep in merged_ids:
                continue
            parent = by_id[keep]
            rec = {"kind": "subtype", "action": "subclass", "parent_id": parent["id"],
                   "parent_label": parent["canonical_label"], "reason": g.get("reason"), "children": []}
            for cid in ids:
                if cid == keep:
                    continue
                child = by_id[cid]
                if child.get("parent") == parent["canonical_label"]:
                    continue                                      # already under this parent — no-op, skip
                rec["children"].append({"id": cid, "label": child["canonical_label"],
                                        "old_parent": child.get("parent")})
                child["parent"] = parent["canonical_label"]       # keep the child type; re-parent it
            if rec["children"]:
                reparents.append(rec)
            continue

        # otherwise MERGE: synonyms always; subtype only when SUBTYPE_MODE == "merge"
        if is_subtype and SUBTYPE_MODE != "merge":
            continue
        members = [by_id[i] for i in ids]
        if is_subtype:
            if keep not in by_id or keep in merged_ids:
                continue
            survivor = by_id[keep]                                # broader parent survives
        else:
            survivor = max(members, key=lambda t: (t.get("support", 0), -int(t["id"][1:])))
        rec = {"kind": kind, "action": "merge", "survivor_id": survivor["id"],
               "survivor_label": survivor["canonical_label"], "reason": g.get("reason"), "merged": [],
               "articles": sorted({a for t in members for a in _articles_of(t)})}
        for m in members:
            if m["id"] == survivor["id"]:
                continue
            for lbl in [m["canonical_label"]] + m.get("aliases", []):
                if lbl != survivor["canonical_label"] and lbl not in survivor["aliases"]:
                    survivor["aliases"].append(lbl)
            survivor["support"] = survivor.get("support", 0) + m.get("support", 0)
            survivor.setdefault("provenance", []).extend(m.get("provenance", []))
            rec["merged"].append({"id": m["id"], "label": m["canonical_label"],
                                  "support": m.get("support", 0), "articles": _articles_of(m)})
            merged_ids.add(m["id"])
        if rec["merged"]:
            rec["resulting_support"] = survivor["support"]
            rec["resulting_aliases"] = list(survivor["aliases"])
            merges.append(rec)

    if merged_ids:
        vocab["types"] = [t for t in vocab["types"] if t["id"] not in merged_ids]

    # write the CONSOLIDATED vocabulary to its own file (raw type_vocabulary.json stays untouched)
    _annotate_vocab_paths(vocab)                 # refresh derived `path`s (re-parenting/merges changed them)
    CONSOLIDATED_VOCAB_PATH.write_text(json.dumps(vocab, indent=2, ensure_ascii=False))

    n_reparented = sum(len(r["children"]) for r in reparents)
    # audit: did the embedder's shortlist actually propose each pair the LLM merged/re-parented?
    def _in_shortlist(a, b):
        return b in _adj.get(a, ()) or a in _adj.get(b, ())
    for _r in merges:
        for _m in _r["merged"]:
            _m["in_shortlist"] = _in_shortlist(_r["survivor_id"], _m["id"])
    for _r in reparents:
        for _c in _r["children"]:
            _c["in_shortlist"] = _in_shortlist(_r["parent_id"], _c["id"])

    report = {
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"), "match_model": MATCH_MODEL,
        "gemini_model": GEMINI_MODEL, "expand_mode": EXPAND_MODE,
        "subtype_mode": SUBTYPE_MODE, "source_vocab": str(VOCAB_PATH.name),
        "consolidated_vocab": str(CONSOLIDATED_VOCAB_PATH.name),
        "consolidate_top_k": CONSOLIDATE_TOP_K, "neighbour_shortlist": shortlist,
        "articles_used": articles_before, "n_articles": len(articles_before),
        "vocab_size_before": len(ents), "vocab_size_after": len(vocab["types"]),
        "n_merges": len(merges), "n_types_removed": len(merged_ids), "n_reparented": n_reparented,
        "merges": merges, "reparents": reparents,
    }
    if write_report:
        CONSOLIDATION_REPORT_PATH.write_text(json.dumps(report, indent=2, ensure_ascii=False))

    if verbose:
        print(f"consolidation (1 call): subtype_mode={SUBTYPE_MODE!r} | {len(merges)} merge-group(s), "
              f"{len(merged_ids)} removed, {n_reparented} re-parented | vocab {len(ents)} -> {len(vocab['types'])}")
        for r in merges:
            print(f"  [merge/{r['kind'][:4]}] keep {r['survivor_id']} {r['survivor_label']!r} <= "
                  f"{[m['label'] for m in r['merged']]}")
        for r in reparents:
            print(f"  [subtype] {[c['label'] for c in r['children']]} <: {r['parent_label']}")
        print(f"  consolidated vocab -> {CONSOLIDATED_VOCAB_PATH}  (raw {VOCAB_PATH.name} untouched)")
        if write_report:
            print(f"  report -> {CONSOLIDATION_REPORT_PATH}")
    return report

# Runs on the RAW incremental vocab and writes the consolidated result to a separate file.
_ = consolidate_vocab(load_vocab())

consolidation (1 call): subtype_mode='subclass' | 2 merge-group(s), 2 removed, 14 re-parented | vocab 92 -> 90
  [merge/syno] keep T0023 'AlertIssuedEvent' <= ['StateOfAlertDeclarationEvent']
  [merge/syno] keep T0063 'MediaCoverageEvent' <= ['MediaMonitoringEvent']
  [subtype] ['EvacuationEvent'] <: DepartureEvent
  [subtype] ['Actress'] <: Actor
  [subtype] ['Mayor', 'SecretaryOfState', 'TreasurySecretary', 'LawEnforcementOfficial', 'Congresswoman', 'President'] <: PublicOfficial
  [subtype] ['VIPEntryEvent', 'CelebrityArrivalEvent'] <: GuestArrivalEvent
  [subtype] ['MediaCaptureEvent'] <: MediaCoverageEvent
  [subtype] ['TrialStartEvent'] <: CeremonyStartEvent
  [subtype] ['TennisTournamentCategory'] <: TennisCompetitionCategory
  [subtype] ['OutbreakStartEvent'] <: DiseaseOutbreakEvent
  consolidated vocab -> /Users/marwah/Documents/Thesis/TriplesExtractionFromWebLLMs/Phase2_LLM_Extraction/results/_global/e2e/type_vocabulary_dedup_consolidated.json  (raw type_vocabulary_dedup.json

### Refresh `*_ext.json` after consolidation → **separate** files — no LLM

Apply the consolidation's `merged_id → survivor` mapping (and subclass re-parentings) to the per-article
files deterministically, but write the result to a **new** `nary_graph_typed_consolidated.json`
per article — the incremental `*_ext.json` is left untouched. Also emit one global
`consolidation_ext_changes.json` that lists **only** the nodes that changed (before → after). Cheaper and
more faithful than `FRESH=True`: it keeps the original type decisions and only reflects the merges /
re-parentings. Run this right after `consolidate_vocab`.

In [ ]:
# Produce a CONSOLIDATED copy of each per-article *_ext.json WITHOUT the LLM — the incremental
# *_ext.json files are left UNTOUCHED. Two effects, applied only to the copy:
#   - MERGED types: nodes pointing at a removed vocab_id are remapped to its survivor.
#   - RE-PARENTED types (subclass mode): the type survives, so we re-sync every vocab-typed node's
#     label / definition / parent from the CONSOLIDATED vocabulary (picks up the new parent).
# `type_is_new_to_vocab` is a BUILD-TIME provenance fact (was the type newly minted when this article
# was processed) — consolidation leaves it untouched, so only genuine merges / re-parentings register
# as changes. Writes, per article, nary_graph_typed_consolidated.json, and ONE global changes
# file (results/consolidation_ext_changes.json) listing ONLY the nodes that changed (before -> after).
# Pure bookkeeping — type DECISIONS are untouched. Run right after consolidate_vocab.
EXT_CONSOLIDATED_NAME = f"nary_graph{_sfx}_typed_consolidated.json"   # per-article refreshed copy
EXT_CHANGES_PATH      = RESULTS_DIR / f"_global/e2e/consolidation_ext_changes{_sfx}.json"   # global "what changed" file
_CHANGED_FIELDS = ("type_vocab_id", "type", "type_definition", "type_parent")   # substantive fields only

def refresh_ext_after_consolidation(report=None, verbose=True):
    if report is None:
        if not CONSOLIDATION_REPORT_PATH.exists():
            print("no consolidation report found — run consolidate_vocab first"); return
        report = json.loads(CONSOLIDATION_REPORT_PATH.read_text())
    remap = {m["id"]: r["survivor_id"] for r in report.get("merges", []) for m in r.get("merged", [])}
    by_id = {t["id"]: t for t in load_consolidated()["types"]}    # survivors + re-parented types

    all_changes, total = [], 0
    for fp in sorted(RESULTS_DIR.glob(f"*/e2e/nary_graph{_sfx}_typed.json")):
        data = json.loads(fp.read_text())                        # the untouched incremental file
        aid  = data.get("article_id", fp.parent.name)
        art_changes = []
        for n in data["entities"]:
            v = n.get("type_vocab_id")
            if not v:
                continue
            s = by_id.get(remap.get(v, v))                        # follow a merge, else same id
            if s is None:
                continue
            old = {k: n.get(k) for k in _CHANGED_FIELDS}
            new = {"type_vocab_id": s["id"], "type": s["canonical_label"],
                   "type_definition": s.get("definition"), "type_parent": s.get("parent")}
            if new != old:
                n.update(new)                                     # mutate the in-memory COPY only
                art_changes.append({"text": n.get("text"),
                                    "type_oie": n.get("type_oie"), "before": old, "after": new})
        # write the consolidated copy for EVERY article (the incremental *_ext.json stays intact)
        data["consolidated"], data["subtype_mode"] = True, SUBTYPE_MODE
        (fp.parent / EXT_CONSOLIDATED_NAME).write_text(json.dumps(data, indent=2, ensure_ascii=False))
        if art_changes:
            all_changes.append({"article_id": aid, "n_changed": len(art_changes), "changes": art_changes})
            total += len(art_changes)
            if verbose:
                print(f"  {aid}: {len(art_changes)} node(s) changed")

    EXT_CHANGES_PATH.write_text(json.dumps({
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"), "subtype_mode": SUBTYPE_MODE,
        "consolidated_vocab": str(CONSOLIDATED_VOCAB_PATH.name),
        "n_articles_changed": len(all_changes), "n_nodes_changed": total, "articles": all_changes,
    }, indent=2, ensure_ascii=False))
    print(f"done — wrote {EXT_CONSOLIDATED_NAME} for each article (incremental *_ext.json untouched)\n"
          f"       {total} changed node(s) across {len(all_changes)} article(s) -> {EXT_CHANGES_PATH.name}")

refresh_ext_after_consolidation()

  art2: 7 node(s) changed
  art4: 1 node(s) changed
done — wrote nary_graph_dedup_typed_consolidated.json for each article (incremental *_ext.json untouched)
       8 changed node(s) across 2 article(s) -> consolidation_ext_changes_dedup.json


## Type hierarchy (schema.org + extension) — raw and consolidated, **separate** files

One indented tree of the whole type space — schema.org classes and the extension (minted) types
together, each tagged `[schema.org]` or `[extension]`, subclasses nested under their parent. Built
**twice**, into separate files so you can compare: `results/type_hierarchy.txt` from the **raw
incremental** vocab, and `results/type_hierarchy_consolidated.txt` from the **consolidated** vocab (so
`subclass` re-parentings and merges appear). Set `INCLUDE_ALL_SCHEMA=True` to include the entire
schema.org tree, or leave it `False` to show just the branches your extension types hang off.

In [ ]:
# Type hierarchy: schema.org classes + extension (vocab) types in ONE indented tree, each tagged
# [schema.org] or [extension]; children (subclasses) are nested under their parent. Parameterized by
# which vocab to draw the extension types from, so the RAW incremental vocab and the CONSOLIDATED vocab
# each get their own separate file:
#   type_hierarchy.txt              <- raw incremental vocab (type_vocabulary.json)
#   type_hierarchy_consolidated.txt <- consolidated vocab   (re-parentings/merges reflected)
#   INCLUDE_ALL_SCHEMA=False -> only the schema.org branches that contain an extension type (the
#                              ancestors of your minted types) + all extension types.
#   INCLUDE_ALL_SCHEMA=True  -> the entire schema.org tree as well (large: ~900+ lines).
INCLUDE_ALL_SCHEMA          = False
HIERARCHY_PATH              = RESULTS_DIR / f"_global/e2e/type_hierarchy{_sfx}.txt"                # raw incremental vocab
HIERARCHY_CONSOLIDATED_PATH = RESULTS_DIR / f"_global/e2e/type_hierarchy{_sfx}_consolidated.txt"   # consolidated vocab

def build_type_hierarchy(vocab, write_path, include_all_schema=INCLUDE_ALL_SCHEMA, show=50):
    from collections import defaultdict
    schema_labels = {c["label"] for c in schema_pool}
    schema_parent = {c["label"]: (c["path"].split(" > ")[-1] if c.get("path") else None) for c in schema_pool}
    ext = {t["canonical_label"]: t for t in vocab["types"]}

    parent_of, kind = {}, {}
    for lbl in schema_labels:
        parent_of[lbl] = schema_parent.get(lbl); kind[lbl] = "schema.org"
    for lbl, t in ext.items():                                   # extension overrides if same label
        parent_of[lbl] = t.get("parent"); kind[lbl] = "extension"

    if include_all_schema:
        included = set(schema_labels) | set(ext)
    else:                                                        # only extensions + their ancestor chains
        included = set(ext)
        for lbl in list(ext):
            cur, seen = parent_of.get(lbl), set()
            while cur and cur not in seen and cur not in included:
                seen.add(cur); included.add(cur); cur = parent_of.get(cur)

    children, roots = defaultdict(list), []
    for lbl in included:
        p = parent_of.get(lbl)
        (children[p].append(lbl) if p in included else roots.append(lbl))
    for p in children:
        children[p].sort()

    lines = []
    def walk(lbl, depth, seen):
        tag   = "[extension]" if kind.get(lbl) == "extension" else "[schema.org]"
        extra = f"  (support {ext[lbl].get('support', 0)})" if lbl in ext else ""
        lines.append("  " * depth + f"{lbl}  {tag}{extra}")
        for c in children.get(lbl, []):
            if c not in seen:
                walk(c, depth + 1, seen | {c})
    for r in sorted(set(roots)):
        walk(r, 0, {r})

    text = "\n".join(lines)
    if write_path:
        write_path.write_text(text)
    print(f"type hierarchy: {len(lines)} nodes ({len(ext)} extension) | "
          f"include_all_schema={include_all_schema} -> {write_path}\n" + "-" * 74)
    print(text if len(lines) <= show else "\n".join(lines[:show]) + f"\n  ... ({len(lines) - show} more — see file)")
    return text

# Raw incremental hierarchy (always available), then the consolidated one (reflects re-parentings/merges).
print("=== RAW incremental vocabulary ===")
build_type_hierarchy(load_vocab(),        HIERARCHY_PATH)
print("\n=== CONSOLIDATED vocabulary ===")
build_type_hierarchy(load_consolidated(), HIERARCHY_CONSOLIDATED_PATH)

=== RAW incremental vocabulary ===
type hierarchy: 110 nodes (92 extension) | include_all_schema=False -> /Users/marwah/Documents/Thesis/TriplesExtractionFromWebLLMs/Phase2_LLM_Extraction/results/_global/e2e/type_hierarchy_dedup.txt
--------------------------------------------------------------------------
Quantity  [schema.org]
  Temperature  [extension]  (support 2)
Thing  [schema.org]
  Actor  [extension]  (support 1)
  Actress  [extension]  (support 1)
  AnniversaryEvent  [extension]  (support 1)
  CharityOrganizationType  [extension]  (support 1)
  Congresswoman  [extension]  (support 1)
  CreativeWork  [schema.org]
    WeatherForecast  [extension]  (support 1)
  Designer  [extension]  (support 1)
  Event  [schema.org]
    AffectedByHeatEvent  [extension]  (support 1)
    AlertIssuedEvent  [extension]  (support 3)
    BracingForHeatEvent  [extension]  (support 1)
    CelebrityArrivalEvent  [extension]  (support 1)
    CelebritySightingEvent  [extension]  (support 1)
    CeremonySt

'Quantity  [schema.org]\n  Temperature  [extension]  (support 2)\nThing  [schema.org]\n  Actor  [extension]  (support 1)\n    Actress  [extension]  (support 1)\n  AnniversaryEvent  [extension]  (support 1)\n  CharityOrganizationType  [extension]  (support 1)\n  CreativeWork  [schema.org]\n    WeatherForecast  [extension]  (support 1)\n  Designer  [extension]  (support 1)\n  Event  [schema.org]\n    AffectedByHeatEvent  [extension]  (support 1)\n    AlertIssuedEvent  [extension]  (support 4)\n    BracingForHeatEvent  [extension]  (support 1)\n    CelebritySightingEvent  [extension]  (support 1)\n    CeremonyStartEvent  [extension]  (support 2)\n      TrialStartEvent  [extension]  (support 1)\n    CharitableDonationEvent  [extension]  (support 1)\n    ContactTracingLaunchEvent  [extension]  (support 1)\n    DepartureEvent  [extension]  (support 1)\n      EvacuationEvent  [extension]  (support 1)\n    Disaster  [extension]  (support 1)\n      DiseaseOutbreakEvent  [extension]  (support 1)